In [9]:
import numpy as np
from sklearn.base import BaseEstimator

class MyDummyClassifier(BaseEstimator) :
    # fit() 메소드는 아무것도 학습하지 않음
    def fit(selt,X,y=None) :
        pass
    
    def predict(self,X) :
        pred = np.zeros((X.shape[0], 1))
        for i in range(X.shape[0]) :
            if X['Sex'].iloc[i] == 1 :
                pred[i] = 0
            else :
                pred[i] = 1
                
        return pred
    
    


In [3]:
from sklearn.preprocessing import LabelEncoder


# Null 처리 함수
def fillna(df) :
    df['Age'].fillna(df['Age'].mean(), inplace=True)
    df['Cabin'].fillna('N', inplace=True)
    df['Embarked'].fillna('N', inplace=True)
    return df

# 불필요한 컬럼 제거 함수
def drop_features(df):
    df.drop(['PassengerId', 'Name', 'Ticket'], axis=1, inplace=True)
    return df

# 레이블 인코딩 함수
def encode_features(df):
    df['Cabin'] = df['Cabin'].str[:1]
    features  = ['Cabin','Embarked', 'Sex']
    for feature in features:
        le = LabelEncoder()
        le.fit(df[feature])
        df[feature] = le.transform(df[feature])
    return df

def transform_features(df):
    df = fillna(df)
    df = drop_features(df)
    df = encode_features(df)
    return df



In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

titanic_df = pd.read_csv('../2장/titanic_train.csv')

y_titanic_df = titanic_df['Survived']
X_titanic_df = titanic_df.drop('Survived',axis=1,inplace=False)

X_titanic_df = transform_features(X_titanic_df)
X_train, X_test, y_train,y_test = train_test_split(X_titanic_df,y_titanic_df,test_size=0.2,random_state=0)

myclf = MyDummyClassifier()
myclf.fit(X_train,y_train)
pred = myclf.predict(X_test)
print('Dummy Classifier 정확도는 {0:.4f}'.format(accuracy_score(y_test,pred)))


Dummy Classifier 정확도는 0.7877


/var/folders/q1/wy16nfjn4sn4s87yr7501dc40000gn/T/ipykernel_53984/738179611.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].mean(), inplace=True)
/var/folders/q1/wy16nfjn4sn4s87yr7501dc40000gn/T/ipykernel_53984/738179611.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always

In [ ]:
from sklearn.datasets import load_digits

class MyFakeClassifier(BaseEstimator) :
    def fit(self,X,y):
        pass
    
    def predict(self,X):
        return np.zeros((len(X),1),dtype=bool)
    
digits = load_digits()

print(digits.data)
print(digits.data.shape)
print(digits.target)
print(digits.target.shape)

[[ 0.  0.  5. ...  0.  0.  0.]
 [ 0.  0.  0. ... 10.  0.  0.]
 [ 0.  0.  0. ... 16.  9.  0.]
 ...
 [ 0.  0.  1. ...  6.  0.  0.]
 [ 0.  0.  2. ... 12.  0.  0.]
 [ 0.  0. 10. ... 12.  1.  0.]]
(1797, 64)
[0 1 2 ... 8 9 8]
(1797,)


In [13]:
digits.target == 7

array([False, False, False, ..., False, False, False], shape=(1797,))

In [14]:
y = (digits.target == 7).astype(int)
X_train, X_test, y_train, y_test = train_test_split(digits.data,y,random_state=11)

In [ ]:
print('레이블 테스트 세트 크기 ', y_test.shape)
print(pd.Series(y_test).value_counts())

fakeclf = MyFakeClassifier()
fakeclf.fit(X_train,y_train)
pred = fakeclf.predict(X_test)
print("모든 예측을 0으로 하여도 정확도는 {:.3f}".format(accuracy_score(y_test,pred)))


레이블 테스트 세트 크기  (450,)
0    405
1     45
Name: count, dtype: int64
모든 예측을 0으로 하여도 정확도는 0.900


**Confusion Matrix**

In [18]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test,pred)

array([[405,   0],
       [ 45,   0]])

### 정밀도(Precision)과 재현율(Recall)

In [19]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

print('정밀도 :' ,precision_score(y_test,pred))
print('재현율 :',recall_score(y_test,pred))

정밀도 : 0.0
재현율 : 0.0


/Users/songbeom/PythonWorkSpace/machineLearning/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


**오차행렬,정확도,정밀도,재현율을 한꺼번에 계산하는 함수 생성**

In [27]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

def get_clf_eval(y_test,pred) :
    confusion = confusion_matrix(y_test,pred)
    accuracy = accuracy_score(y_test,pred)
    precision = precision_score(y_test,pred)
    recall = recall_score(y_test,pred)
    print('오차 행렬')
    print(confusion)
    print('정확도: {0:.4f}, 정밀도: {1:.4f}, 재현율: {2:.4f}'.format(accuracy,precision,recall))
    

In [28]:
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

titanic_df = pd.read_csv('../2장/titanic_train.csv')
y_titanic_df = titanic_df['Survived']
X_titanic_df = titanic_df.drop('Survived',axis=1,inplace=False)
X_titanic_df = transform_features(X_titanic_df)
X_train, X_test, y_train, y_test = train_test_split(X_titanic_df,y_titanic_df,test_size=0.2,random_state=11)

lr_clf = LogisticRegression(solver='liblinear')

lr_clf.fit(X_train,y_train)
pred= lr_clf.predict(X_test)
get_clf_eval(y_test,pred)

오차 행렬
[[108  10]
 [ 14  47]]
정확도: 0.8659, 정밀도: 0.8246, 재현율: 0.7705


### Precision/Recall Trade-Off

**predict_proba()메소드 확인**

In [34]:
pred_proba = lr_clf.predict_proba(X_test)
pred = lr_clf.predict(X_test)

print('predict_proba() 결과 Shape :{0}'.format(pred_proba.shape))
print('array에서 앞에 3개만 추출 : \n',pred_proba[:3])

pred_proba_result = np.concatenate([pred_proba,pred.reshape(-1,1)],axis=1)
print('두개의 class 중에서 더 큰 확률을 클래스 값으로 예측 \n',pred_proba_result[:3])

predict_proba() 결과 Shape :(179, 2)
array에서 앞에 3개만 추출 : 
 [[0.44935225 0.55064775]
 [0.86335511 0.13664489]
 [0.86429643 0.13570357]]
두개의 class 중에서 더 큰 확률을 클래스 값으로 예측 
 [[0.44935225 0.55064775 1.        ]
 [0.86335511 0.13664489 0.        ]
 [0.86429643 0.13570357 0.        ]]
